# RTFM Full Pipeline: Detect → Sample → Explain → Judge

End-to-end pipeline on Colab GPU:

1. **RTFM Inference** — Score all 199 test videos (GPU)
2. **Video-Level Gate** — Flag anomalous videos (`score_abnormal > τ`)
3. **Segment Detection** — Find contiguous anomalous snippet runs
4. **Smart Sampling** — Select key frames (onset + resolution + score-weighted)
5. **Frame Extraction** — Pull actual frames from video directories
6. **VLM Explanation** — GPT-4o describes the anomaly from sampled frames
7. **Judge** — GPT-4o-as-judge scores AI explanation vs human ground truth

### Prerequisites on Google Drive
```
MyDrive/
  rtfm_colab.zip              # I3D features (SH_Test + SH_Train) + list files
  rtfm_src/                    # model.py, option.py, utils.py, dataset.py, config.py
  rtfm_checkpoints/
    rtfm_best.pkl              # Trained RTFM checkpoint
  shanghai_pipeline_upload.zip # Frames zip (uploaded from Desktop)
  annotations.json             # Human ground-truth explanations
```

**First time setup:** After uploading `shanghai_pipeline_upload.zip`, unzip it on Colab:
```python
!unzip -q /content/drive/MyDrive/shanghai_pipeline_upload.zip -d /content/drive/MyDrive/shanghai_frames
```
This creates `MyDrive/shanghai_frames/frames/01_0015/`, etc. Cell 3 auto-detects the path.

In [30]:
# ── Cell 1: GPU Check ────────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-80GB
Memory  : 85.1 GB


In [35]:
# ── Cell 2: Mount Drive + Setup ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, sys, json, importlib, importlib.util, time
import numpy as np
import cv2
from pathlib import Path

os.makedirs('/content/rtfm', exist_ok=True)

# Unzip I3D features
if not os.path.exists('/content/rtfm/data'):
    !unzip -q /content/drive/MyDrive/rtfm_colab.zip -d /content/rtfm
    print('Unzipped I3D features.')
else:
    print('I3D features already present.')

# Copy RTFM source files
src_dir = '/content/drive/MyDrive/rtfm_src'
for fname in ['model.py', 'dataset.py', 'option.py', 'utils.py', 'config.py']:
    dst = f'/content/rtfm/{fname}'
    src = f'{src_dir}/{fname}'
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy2(src, dst)

# Fix paths in test list
list_path = '/content/rtfm/list/shanghai-i3d-test-10crop.list'
with open(list_path) as f:
    lines = f.readlines()
fixed = [f'/content/rtfm/data/SH_Test_ten_crop_i3d/{l.strip().split("/")[-1]}\n' for l in lines]
with open(list_path, 'w') as f:
    f.writelines(fixed)
print(f'Test list: {len(fixed)} entries')

# Copy checkpoint
os.makedirs('/content/rtfm/ckpt', exist_ok=True)
ckpt_src = '/content/drive/MyDrive/rtfm_checkpoints/rtfm_best.pkl'
ckpt_dst = '/content/rtfm/ckpt/rtfm_best.pkl'
if not os.path.exists(ckpt_dst):
    shutil.copy2(ckpt_src, ckpt_dst)

# Load RTFM modules
os.chdir('/content/rtfm')
sys.path.insert(0, '/content/rtfm')

def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

option  = _load_module('option',  '/content/rtfm/option.py')
model_m = _load_module('model',   '/content/rtfm/model.py')

args   = option.parser.parse_args([])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

net = model_m.Model(args.feature_size, args.batch_size)
net.load_state_dict(torch.load(ckpt_dst, map_location=device))
net = net.to(device).eval()
print(f'Model loaded on {device}  (k_abn={net.k_abn})')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
I3D features already present.
Test list: 199 entries
Model loaded on cuda  (k_abn=3)


In [36]:
# ── Cell 2b: Unzip frames (run once, skip if already done) ───────────────────
import os
from pathlib import Path

zip_path = '/content/drive/MyDrive/shanghai_pipeline_upload.zip'
target   = '/content/drive/MyDrive/shanghai_frames'

# Check if already unzipped
already_done = False
for check in [f'{target}/frames', target]:
    p = Path(check)
    if p.exists() and len(list(p.iterdir())) > 50:
        already_done = True
        print(f'Frames already unzipped at: {check}  ({len(list(p.iterdir()))} dirs)')
        break

if not already_done:
    if os.path.exists(zip_path):
        print(f'Unzipping {zip_path} → {target}/ ...')
        !unzip -q -n "{zip_path}" -d "{target}"
        print('Done.')
    else:
        print(f'WARNING: {zip_path} not found on Drive.')
        print('Upload shanghai_pipeline_upload.zip to MyDrive/ first.')

Frames already unzipped at: /content/drive/MyDrive/shanghai_frames/frames  (174 dirs)


In [37]:
# ── Cell 3: Configure paths (auto-detect frames location) ───────────────────

ANNOTATIONS_PATH = Path('/content/drive/MyDrive/annotations.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/rtfm_pipeline_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect frames directory — handles different unzip structures
FRAMES_DIR = None
candidates = [
    Path('/content/drive/MyDrive/shanghai_frames'),           # direct upload
    Path('/content/drive/MyDrive/shanghai_frames/frames'),    # unzipped with nested frames/
    Path('/content/drive/MyDrive/frames'),                    # unzipped at root
    Path('/content/drive/MyDrive/SHANGHAI_Test/frames'),      # original structure
    Path('/content/drive/MyDrive/shanghai_pipeline_upload/frames'),  # alt unzip name
]

for cand in candidates:
    if not cand.exists():
        continue
    subdirs = [d for d in cand.iterdir() if d.is_dir()]
    # Descend one level if this dir only contains a single 'frames' subfolder
    if len(subdirs) == 1 and subdirs[0].name == 'frames':
        cand = subdirs[0]
        subdirs = [d for d in cand.iterdir() if d.is_dir()]
    if len(subdirs) > 50:
        FRAMES_DIR = cand
        break

if FRAMES_DIR is None:
    # Last resort: search for any directory containing 01_0015 (known test video)
    import subprocess
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-maxdepth', '4', '-type', 'd', '-name', '01_0015'],
        capture_output=True, text=True, timeout=30
    )
    if result.stdout.strip():
        found = Path(result.stdout.strip().split('\n')[0])
        FRAMES_DIR = found.parent
        print(f'Auto-detected frames at: {FRAMES_DIR}')

if FRAMES_DIR and FRAMES_DIR.exists():
    frame_dirs = sorted([d.name for d in FRAMES_DIR.iterdir() if d.is_dir()])
    print(f'Frames directory: {FRAMES_DIR}')
    print(f'  Found {len(frame_dirs)} video directories')
    print(f'  Examples: {frame_dirs[:5]}')
else:
    print(f'ERROR: Could not find frames directory on Drive!')
    print('Searched:', [str(c) for c in candidates])
    print('Please set FRAMES_DIR manually in this cell.')
    FRAMES_DIR = Path('/content/drive/MyDrive/shanghai_frames')  # fallback

# Verify annotations
if ANNOTATIONS_PATH.exists():
    with open(ANNOTATIONS_PATH) as f:
        annotations_raw = json.load(f)
    annotations = {e['video_id']: e for e in annotations_raw if 'video_id' in e}
    print(f'Annotations: {len(annotations)} videos with ground truth')
else:
    print(f'WARNING: annotations.json not found at {ANNOTATIONS_PATH}')
    print('Upload data/SHANGHAI/anomalous_videos/annotations.json to MyDrive/')
    annotations = {}

Frames directory: /content/drive/MyDrive/shanghai_frames/frames
  Found 174 video directories
  Examples: ['01_001', '01_0015', '01_0025', '01_0027', '01_0028']


OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/annotations.json'

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1+2: RTFM INFERENCE + VIDEO-LEVEL GATE
# ══════════════════════════════════════════════════════════════════════════════

GATE_THRESHOLD    = 0.2   # score_abnormal > this → flagged
SEGMENT_THRESHOLD = 0.3   # per-snippet score > this → anomalous segment
FRAME_BUDGET      = 8     # max frames to sample per video
MIN_GAP           = 2     # min snippet spacing between selected frames

with open('/content/rtfm/list/shanghai-i3d-test-10crop.list') as f:
    test_paths = [l.strip() for l in f if l.strip()]

print(f'Test videos      : {len(test_paths)}')
print(f'Gate threshold   : {GATE_THRESHOLD}')
print(f'Segment threshold: {SEGMENT_THRESHOLD}')
print(f'Frame budget     : {FRAME_BUDGET}')
print()

results = {}
n_flagged = 0
n_skipped = 0

with torch.no_grad():
    for idx, path in enumerate(test_paths):
        video_name = os.path.basename(path).replace('.npy', '').replace('_i3d', '')

        features = np.load(path, allow_pickle=True).astype(np.float32)
        inp = torch.from_numpy(features).unsqueeze(0).to(device)
        inp = inp.permute(0, 2, 1, 3)

        score_abnormal, _, _, _, _, _, logits, _, _, _ = net(inputs=inp)

        logits_squeezed = torch.squeeze(logits, 1)
        seg_scores = torch.mean(logits_squeezed, 0).squeeze().cpu().numpy()
        n_segments = len(seg_scores)
        if seg_scores.ndim == 0:
            seg_scores = seg_scores.reshape(1)

        gate_score = float(score_abnormal.squeeze().cpu())

        if gate_score <= GATE_THRESHOLD:
            n_skipped += 1
            results[video_name] = {
                'segment_scores': seg_scores.tolist(),
                'n_segments': n_segments,
                'gate_score': gate_score,
                'flagged': False,
            }
            continue

        n_flagged += 1
        results[video_name] = {
            'segment_scores': seg_scores.tolist(),
            'n_segments': n_segments,
            'gate_score': gate_score,
            'flagged': True,
        }

        if (idx + 1) % 50 == 0 or (idx + 1) == len(test_paths):
            print(f'  [{idx+1:3d}/{len(test_paths)}]  flagged={n_flagged}  skipped={n_skipped}')

print(f'\nDone. Total={len(results)}  Flagged={n_flagged}  Skipped={n_skipped}')

Test videos      : 199
Gate threshold   : 0.2
Segment threshold: 0.3
Frame budget     : 8


Done. Total=199  Flagged=46  Skipped=153


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3+4: SEGMENT DETECTION + SMART FRAME SAMPLING
# ══════════════════════════════════════════════════════════════════════════════

def find_anomalous_segments(seg_scores, threshold):
    """Find contiguous runs where score > threshold."""
    segments = []
    in_seg = False
    start = 0
    for i, s in enumerate(seg_scores):
        if not in_seg and s > threshold:
            in_seg = True
            start = i
        elif in_seg and s <= threshold:
            segments.append((start, i - 1))
            in_seg = False
    if in_seg:
        segments.append((start, len(seg_scores) - 1))
    return segments


def sample_snippets(segments, seg_scores, budget, min_gap):
    """Smart snippet selection: onset+resolution + score-weighted fill."""
    if not segments:
        return []
    total_len = sum(e - s + 1 for s, e in segments)
    all_selected = []

    for seg_s, seg_e in segments:
        seg_len = seg_e - seg_s + 1
        seg_budget = max(2, round(budget * seg_len / total_len))

        selected = {seg_s, seg_e}
        remaining = seg_budget - len(selected)

        if remaining > 0 and seg_len > 2:
            candidates = sorted(
                [(i, seg_scores[i]) for i in range(seg_s + 1, seg_e)],
                key=lambda x: x[1], reverse=True
            )
            for c_idx, c_score in candidates:
                if remaining <= 0:
                    break
                if any(abs(c_idx - s) < min_gap for s in selected):
                    continue
                selected.add(c_idx)
                remaining -= 1

        for s_idx in sorted(selected):
            all_selected.append({'snippet_idx': s_idx, 'score': float(seg_scores[s_idx])})

    return all_selected


def find_segments_adaptive(seg_scores, threshold, budget, min_gap):
    """
    Adaptive segment threshold: start at `threshold`, lower in steps until
    we can fill at least half the frame budget — ensures weak anomalies
    still get enough frames to give GPT-4o useful context.

    Steps tried: threshold → 0.67x → 0.33x → 0.05 → top-k fallback
    """
    steps = [threshold, threshold * 0.67, threshold * 0.33, 0.05]
    for t in steps:
        segs = find_anomalous_segments(seg_scores, t)
        if not segs:
            continue
        selected = sample_snippets(segs, seg_scores, budget, min_gap)
        if len(selected) >= max(2, budget // 2):
            return segs, selected, round(t, 3)
    # absolute fallback: top-k snippets by score, treat as one segment
    ranked = sorted(range(len(seg_scores)), key=lambda i: seg_scores[i], reverse=True)
    top_k = min(budget, len(ranked))
    top_indices = sorted(ranked[:top_k])
    segs = [(top_indices[0], top_indices[-1])] if top_indices else []
    selected = sample_snippets(segs, seg_scores, budget, min_gap)
    return segs, selected, 0.0


# Process all flagged videos
flagged_videos = {}

for vname, vdata in results.items():
    if not vdata['flagged']:
        continue

    scores = np.array(vdata['segment_scores'])
    segments, selected, used_thr = find_segments_adaptive(
        scores, SEGMENT_THRESHOLD, FRAME_BUDGET, MIN_GAP
    )

    flagged_videos[vname] = {
        **vdata,
        'anomalous_segments': [(s, e) for s, e in segments],
        'selected_snippets': selected,
        'used_threshold': used_thr,
    }

# Print all 46 flagged videos with threshold info
print(f'Flagged videos: {len(flagged_videos)}  '
      f'(segment_threshold={SEGMENT_THRESHOLD}, adaptive fallback enabled)\n')
print(f'  {"Video":<14s} {"Gate":>6s} {"Segs":>5s} {"Sel":>4s} {"Thr":>6s}')
print(f'  {"-"*44}')
for vname, vdata in sorted(flagged_videos.items()):
    n_sel = len(vdata["selected_snippets"])
    n_seg = len(vdata["anomalous_segments"])
    thr   = vdata.get("used_threshold", SEGMENT_THRESHOLD)
    note  = " ↓lowered" if thr < SEGMENT_THRESHOLD else ""
    print(f'  {vname:<14s} {vdata["gate_score"]:>6.3f} {n_seg:>5d} {n_sel:>4d} {thr:>6.3f}{note}')

Flagged videos: 46  (segment_threshold=0.3, adaptive fallback enabled)

  Video            Gate  Segs  Sel    Thr
  --------------------------------------------
  01_0015         0.664     1    4  0.300
  01_0025         1.000     1    8  0.300
  01_0027         0.646     1    8  0.300
  01_0028         0.409     2    7  0.300
  01_0051         0.959     1    8  0.300
  01_0052         0.776     1    7  0.300
  01_0053         0.998     1    8  0.300
  01_0056         0.617     1    4  0.300
  01_0135         0.998     1    7  0.300
  01_0136         0.595     1    5  0.300
  01_0139         0.964     1    5  0.300
  01_0162         0.305     2    4  0.201 ↓lowered
  01_0177         0.868     1    5  0.300
  01_074          0.812     1    4  0.300
  02_0161         0.896     1    7  0.300
  03_0031         0.577     2    8  0.300
  03_0032         0.998     1    6  0.300
  03_0033         0.725     1    5  0.300
  03_0059         0.596     1    6  0.300
  04_0004         0.989     4   

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5: FRAME EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════
FRAMES_DIR = Path('/content/drive/MyDrive/shanghai_frames')
def snippet_to_frame_num(snippet_idx, total_frames, n_segments):
    """Map snippet index to the middle frame of that snippet's range."""
    frames_per_snippet = total_frames / n_segments
    start_frame = int(snippet_idx * frames_per_snippet)
    mid_frame = start_frame + int(frames_per_snippet // 2)
    return min(mid_frame, total_frames - 1)


def extract_frame(frame_dir, frame_num):
    """Read a frame from directory (supports 3-digit and 4-digit naming)."""
    for fmt in [f'{frame_num:03d}.jpg', f'{frame_num:04d}.jpg']:
        p = frame_dir / fmt
        if p.exists():
            img = cv2.imread(str(p))
            if img is not None:
                return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return None


# ── Sanity-check FRAMES_DIR before extracting ────────────────────────────────
print(f'FRAMES_DIR : {FRAMES_DIR}')
print(f'  exists   : {FRAMES_DIR.exists()}')
if FRAMES_DIR.exists():
    subdirs = sorted([d.name for d in FRAMES_DIR.iterdir() if d.is_dir()])
    print(f'  sample   : {subdirs[:5]}')
    # Descend into a single 'frames' subfolder if that's all that's here
    if subdirs == ['frames'] or (len(subdirs) == 1 and subdirs[0] == 'frames'):
        FRAMES_DIR = FRAMES_DIR / 'frames'
        subdirs = sorted([d.name for d in FRAMES_DIR.iterdir() if d.is_dir()])
        print(f'  ↳ descended into frames/: {FRAMES_DIR}')
        print(f'  sample   : {subdirs[:5]}')
else:
    # Drive may need to be remounted — try to find frames anywhere under MyDrive
    import subprocess
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-maxdepth', '5',
         '-type', 'd', '-name', '01_0015'],
        capture_output=True, text=True, timeout=60
    )
    hit = result.stdout.strip()
    if hit:
        FRAMES_DIR = Path(hit.split('\n')[0]).parent
        print(f'  ↳ auto-fixed to: {FRAMES_DIR}')
    else:
        raise FileNotFoundError(
            f'Cannot find frames under /content/drive/MyDrive. '
            f'Make sure Drive is mounted (Cell 2) and the zip was unzipped (Cell 2b).'
        )

# Verify the first flagged video can be found
first_vname = next(iter(flagged_videos))
print(f'  test dir : {FRAMES_DIR / first_vname}  '
      f'exists={( FRAMES_DIR / first_vname).exists()}')
print()

# Build a lookup: lowercase dir name → actual Path  (handles case/zero-pad mismatches)
available_dirs = {d.name.lower(): d for d in FRAMES_DIR.iterdir() if d.is_dir()}

def resolve_frame_dir(vname):
    """Return the frame directory for vname, trying exact then fuzzy matches."""
    exact = FRAMES_DIR / vname
    if exact.exists():
        return exact
    # Try lowercase match
    if vname.lower() in available_dirs:
        return available_dirs[vname.lower()]
    # Try zero-padding the numeric part: 01_074 → 01_0074
    parts = vname.split('_')
    if len(parts) == 2:
        padded = f'{parts[0]}_{parts[1].zfill(4)}'
        if padded.lower() in available_dirs:
            return available_dirs[padded.lower()]
        # Also try stripping leading zeros: 01_0074 → 01_074
        stripped = f'{parts[0]}_{parts[1].lstrip("0") or "0"}'
        if stripped.lower() in available_dirs:
            return available_dirs[stripped.lower()]
    return None

# Extract frames for all flagged videos
extraction_results = {}
n_extracted = 0
n_missing = 0
missing_list = []

for vname, vdata in flagged_videos.items():
    frame_dir = resolve_frame_dir(vname)
    if frame_dir is None:
        n_missing += 1
        missing_list.append(vname)
        continue

    total_frames = len(list(frame_dir.glob('*.jpg')))
    if total_frames == 0:
        n_missing += 1
        continue

    n_segments = vdata['n_segments']
    frames = []

    # Save extracted frames to output dir
    vid_out = OUTPUT_DIR / vname
    vid_out.mkdir(parents=True, exist_ok=True)

    for snip in vdata['selected_snippets']:
        fnum = snippet_to_frame_num(snip['snippet_idx'], total_frames, n_segments)
        img = extract_frame(frame_dir, fnum)
        fname = None
        if img is not None:
            fname = f"snippet_{snip['snippet_idx']:03d}_frame_{fnum:04d}.jpg"
            bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(vid_out / fname), bgr)

        frames.append({
            'snippet_idx': snip['snippet_idx'],
            'score': snip['score'],
            'frame_num': fnum,
            'file': fname,
            'image': img,
        })

    # Save metadata
    meta = {
        'video_id': vname,
        'gate_score': vdata['gate_score'],
        'n_segments': n_segments,
        'segment_scores': vdata['segment_scores'],
        'anomalous_segments': [{'start': s, 'end': e} for s, e in vdata['anomalous_segments']],
        'extracted_frames': [
            {k: v for k, v in f.items() if k != 'image'}
            for f in frames
        ],
    }
    with open(vid_out / 'metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)

    extraction_results[vname] = {'meta': meta, 'frames': frames}
    n_extracted += 1

print(f'Extracted frames for {n_extracted} videos')
if n_missing > 0:
    print(f'Missing frame dirs for {n_missing} videos (check FRAMES_DIR)')

FRAMES_DIR : /content/drive/MyDrive/shanghai_frames
  exists   : True
  sample   : ['frames']
  ↳ descended into frames/: /content/drive/MyDrive/shanghai_frames/frames
  sample   : ['01_001', '01_0015', '01_0025', '01_0027', '01_0028']
  test dir : /content/drive/MyDrive/shanghai_frames/frames/01_0015  exists=True

Extracted frames for 34 videos
Missing frame dirs for 12 videos (check FRAMES_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 6: VLM EXPLANATION (GPT-4o)
# ══════════════════════════════════════════════════════════════════════════════

!pip install -q openai

import base64
from openai import OpenAI

# ── Load API key (tries Colab Secrets first, then env, then manual fallback) ──
api_key = None

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key:
        print('API key loaded from Colab Secrets.')
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('OPENAI_API_KEY', '')

if not api_key:
    # ── MANUAL FALLBACK: paste key here if Colab Secrets not set up ──────────
    api_key = os.environ.get('OPENAI_API_KEY', '')
    # ─────────────────────────────────────────────────────────────────────────

if not api_key:
    raise ValueError(
        'No OpenAI API key found!\n'
        'Fix: In Colab, click the 🔑 key icon (left sidebar) → Secrets → '
        'add OPENAI_API_KEY with your sk-... value, then re-run this cell.'
    )

os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)
print(f'OpenAI client ready.  (key: {api_key[:8]}...{api_key[-4:]})')

In [ ]:
# ── VLM Prompt + Helper Functions ─────────────────────────────────────────────

SYSTEM_PROMPT = """You are a surveillance video anomaly analyst. You will be shown a set of \
frames sampled from a surveillance video that has been flagged as anomalous \
by a weakly-supervised anomaly detection model (RTFM).

The frames are ordered temporally. Each frame comes from a specific temporal \
snippet of the video, and you are given the anomaly score for that snippet \
(0 = normal, 1 = highly anomalous).

The frames were specifically selected from the anomalous portions of the \
video — they represent the onset, peak, and resolution of the detected anomaly.

Your task: Based on ALL the frames and their anomaly scores together, provide \
a single concise explanation (2-3 sentences) of what anomalous activity is \
happening. Focus on:
- WHAT is happening (the specific anomalous activity)
- WHO/WHAT is involved (people, vehicles, objects — describe appearance)
- WHEN in the sequence it starts and ends
- WHY it is anomalous (how it deviates from normal pedestrian behaviour)

Respond with ONLY a JSON object in this exact format:
{"explanation": "..."}"""


def encode_b64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')


def build_vlm_message(vname, meta, vid_out_dir):
    """Build multi-modal message with frames + scores."""
    frames = meta['extracted_frames']
    segs = meta['anomalous_segments']
    seg_str = ', '.join(f"snippets {s['start']}-{s['end']}" for s in segs)
    score_str = ', '.join(f"snippet {f['snippet_idx']}={f['score']:.3f}" for f in frames)

    text = (
        f"This video has {meta['n_segments']} temporal snippets (~16 frames each). "
        f"Anomalous segments: [{seg_str}]. "
        f"Video-level gate score: {meta['gate_score']:.3f}. "
        f"Below are {len(frames)} frames from the anomalous segments. "
        f"Per-snippet scores: [{score_str}]."
    )
    content = [{'type': 'text', 'text': text}]

    for f in frames:
        if f.get('file') is None:
            continue
        img_path = vid_out_dir / f['file']
        if not img_path.exists():
            continue
        content.append({'type': 'text', 'text': f"Frame from snippet {f['snippet_idx']} (frame #{f['frame_num']}, score: {f['score']:.3f}):"})
        b64 = encode_b64(str(img_path))
        content.append({
            'type': 'image_url',
            'image_url': {'url': f'data:image/jpeg;base64,{b64}', 'detail': 'high'},
        })

    return content


def query_vlm(content):
    try:
        resp = client.chat.completions.create(
            model='gpt-4o',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': content},
            ],
            max_tokens=300,
            response_format={'type': 'json_object'},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print(f'    ERROR: {e}')
        return {'explanation': f'ERROR: {e}'}

print('VLM functions defined.')

In [ ]:
# ── Generate explanations for all extracted videos ─────────────────────────────

explanations = {}

for i, (vname, edata) in enumerate(sorted(extraction_results.items())):
    meta = edata['meta']
    vid_out = OUTPUT_DIR / vname

    valid_frames = [f for f in meta['extracted_frames'] if f.get('file')]
    if not valid_frames:
        continue

    print(f'[{i+1}/{len(extraction_results)}] {vname}  '
          f'gate={meta["gate_score"]:.3f}  frames={len(valid_frames)} ...', end=' ', flush=True)

    content = build_vlm_message(vname, meta, vid_out)
    response = query_vlm(content)
    explanation = response.get('explanation', '')

    print(f'-> "{explanation[:80]}..."' if len(explanation) > 80 else f'-> "{explanation}"')

    result = {
        'video_id': vname,
        'gate_score': meta['gate_score'],
        'n_frames_sent': len(valid_frames),
        'explanation': explanation,
    }
    explanations[vname] = result

    with open(vid_out / 'explanation.json', 'w') as f:
        json.dump(result, f, indent=2)

    time.sleep(0.5)

# Save summary
with open(OUTPUT_DIR / 'explanations_summary.json', 'w') as f:
    json.dump(list(explanations.values()), f, indent=2)

print(f'\nDone. {len(explanations)} explanations generated.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 7: JUDGE (GPT-4o-as-judge vs Human Ground Truth)
# ══════════════════════════════════════════════════════════════════════════════

JUDGE_PROMPT = """You are an impartial judge evaluating the quality of an AI-generated \
explanation of an anomalous event in a surveillance video.

You will be given:
1. A HUMAN ground-truth explanation written by someone who watched the full video.
2. An AI-GENERATED explanation produced by a vision-language model that only \
saw a few sampled frames guided by anomaly scores from a weakly-supervised model.

Score the AI explanation on these 4 criteria (each 1-5):

- **correctness**: Does the AI identify the same anomaly as the human? \
(1 = completely wrong anomaly, 3 = partially correct, 5 = exact same anomaly)
- **specificity**: Does the AI mention specific details (objects, people, \
actions, clothing)? (1 = very vague, 5 = rich detail)
- **completeness**: Does the AI capture all aspects the human mentioned? \
(1 = misses everything, 5 = covers all key points)
- **fluency**: Is the AI explanation well-written and clear? \
(1 = incoherent, 5 = natural and clear)

Also provide a brief justification (1-2 sentences) for your scores.

Respond with ONLY a JSON object:
{"correctness": 1-5, "specificity": 1-5, "completeness": 1-5, "fluency": 1-5, "justification": "..."}"""


def query_judge(human_expl, ai_expl):
    user_msg = (
        f'HUMAN ground-truth explanation:\n"{human_expl}"\n\n'
        f'AI-generated explanation:\n"{ai_expl}"'
    )
    try:
        resp = client.chat.completions.create(
            model='gpt-4o',
            messages=[
                {'role': 'system', 'content': JUDGE_PROMPT},
                {'role': 'user', 'content': user_msg},
            ],
            max_tokens=300,
            response_format={'type': 'json_object'},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print(f'    ERROR: {e}')
        return {'correctness': None, 'specificity': None,
                'completeness': None, 'fluency': None,
                'justification': f'ERROR: {e}'}

print('Judge function defined.')

In [ ]:
# ── Run judge on all 46 explained videos ──────────────────────────────────────
#
# Ground truth assignment:
#   4-digit video ID  → truly anomalous → use specific annotation explanation
#   3-digit video ID  → normal video (false positive gate) → use normal GT below

NORMAL_GT = 'There is nothing anomalous in this video. All pedestrians are walking normally.'

judge_results = []
n_anomalous_judged = 0
n_normal_judged = 0

for i, (vname, expl_data) in enumerate(sorted(explanations.items())):
    ai_expl = expl_data.get('explanation', '')
    if not ai_expl or ai_expl.startswith('ERROR'):
        continue

    # Determine if this is a true anomaly (4-digit) or normal FP (3-digit)
    clip_id = vname.split('_')[1] if '_' in vname else vname
    is_truly_anomalous = len(clip_id) == 4

    if is_truly_anomalous:
        human_data = annotations.get(vname)
        if not human_data:
            print(f'  SKIP {vname}: no annotation found')
            continue
        human_expl = human_data['explanation']
        gt_start = human_data.get('anomaly_start_frame')
        gt_end   = human_data.get('anomaly_end_frame')
        video_type = 'anomalous'
        n_anomalous_judged += 1
    else:
        # Normal video falsely flagged by gate
        human_expl = NORMAL_GT
        gt_start = None
        gt_end   = None
        video_type = 'normal_FP'
        n_normal_judged += 1

    print(f'[{i+1}] {vname}  [{video_type}]')
    print(f'  AI : "{ai_expl[:90]}..."' if len(ai_expl) > 90 else f'  AI : "{ai_expl}"')
    print(f'  GT : "{human_expl[:90]}..."' if len(human_expl) > 90 else f'  GT : "{human_expl}"')

    scores = query_judge(human_expl, ai_expl)
    print(f'  -> C={scores.get("correctness")} S={scores.get("specificity")} '
          f'Co={scores.get("completeness")} F={scores.get("fluency")}')
    print()

    result = {
        'video_id': vname,
        'video_type': video_type,
        'human_explanation': human_expl,
        'ai_explanation': ai_expl,
        'gate_score': expl_data.get('gate_score'),
        'gt_anomaly_start': gt_start,
        'gt_anomaly_end': gt_end,
        'scores': scores,
    }
    judge_results.append(result)

    vid_out = OUTPUT_DIR / vname
    with open(vid_out / 'judge.json', 'w') as f:
        json.dump(result, f, indent=2)

    time.sleep(0.5)

# Save full summary
with open(OUTPUT_DIR / 'judge_summary.json', 'w') as f:
    json.dump(judge_results, f, indent=2)

print(f'Judged {len(judge_results)} videos total  '
      f'({n_anomalous_judged} anomalous + {n_normal_judged} normal false positives)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

if not judge_results:
    print('No judge results to summarize.')
else:
    def print_group_summary(label, group):
        if not group:
            print(f'\n  No results for: {label}')
            return
        metrics = {'correctness': [], 'specificity': [], 'completeness': [], 'fluency': []}
        for r in group:
            for key in metrics:
                val = r['scores'].get(key)
                if val is not None:
                    metrics[key].append(val)

        print(f'\n{"=" * 65}')
        print(f'  {label}  (n={len(group)})')
        print(f'{"=" * 65}')
        print(f'  {"Metric":<15s} {"Mean":>6s} {"Std":>6s} {"Min":>5s} {"Max":>5s}')
        print(f'  {"-"*48}')
        all_vals = []
        for key in metrics:
            vals = metrics[key]
            if vals:
                arr = np.array(vals)
                print(f'  {key:<15s} {arr.mean():>6.2f} {arr.std():>6.2f} '
                      f'{arr.min():>5.1f} {arr.max():>5.1f}')
                all_vals.extend(vals)
        if all_vals:
            overall = np.array(all_vals)
            print(f'  {"-"*48}')
            print(f'  {"OVERALL":<15s} {overall.mean():>6.2f} {overall.std():>6.2f} '
                  f'{overall.min():>5.1f} {overall.max():>5.1f}')
        print(f'{"=" * 65}')

        print(f'\n  {"Video":<14s} {"Type":<12s} {"C":>3s} {"S":>3s} {"Co":>3s} {"F":>3s}  Justification')
        print(f'  {"-"*80}')
        for r in sorted(group, key=lambda x: x['scores'].get('correctness', 0), reverse=True):
            s = r['scores']
            vtype = '⚠ FP-normal' if r['video_type'] == 'normal_FP' else 'anomalous'
            print(f'  {r["video_id"]:<14s} {vtype:<12s} '
                  f'{s.get("correctness","?"):>3} {s.get("specificity","?"):>3} '
                  f'{s.get("completeness","?"):>3} {s.get("fluency","?"):>3}  '
                  f'{s.get("justification","")[:55]}')

    anomalous_results = [r for r in judge_results if r['video_type'] == 'anomalous']
    normal_fp_results = [r for r in judge_results if r['video_type'] == 'normal_FP']

    print_group_summary('ANOMALOUS VIDEOS — AI explanation vs specific ground truth', anomalous_results)
    print_group_summary('NORMAL FALSE POSITIVES — AI explanation vs "nothing anomalous"', normal_fp_results)

    # Combined overall
    all_vals = []
    for r in judge_results:
        for key in ['correctness', 'specificity', 'completeness', 'fluency']:
            val = r['scores'].get(key)
            if val is not None:
                all_vals.append(val)
    if all_vals:
        arr = np.array(all_vals)
        print(f'\n  COMBINED ALL 46 VIDEOS: mean={arr.mean():.2f}  std={arr.std():.2f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GATE ACCURACY (same as score_segments_colab.ipynb Cell 5)
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

# Ground truth: 4-digit clip IDs are anomalous, 3-digit are normal
gt_labels = []
gate_scores = []

for vname, vdata in results.items():
    parts = vname.split('_')
    clip_id = parts[1] if len(parts) >= 2 else parts[0]
    gt_labels.append(1 if len(clip_id) == 4 else 0)
    gate_scores.append(vdata['gate_score'])

gt_labels = np.array(gt_labels)
gate_scores = np.array(gate_scores)

preds = (gate_scores > GATE_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(gt_labels, preds).ravel()

print(f'=== Video-Level Gate (threshold = {GATE_THRESHOLD}) ===')
print(f'  GT: {gt_labels.sum()} anomalous, {(gt_labels==0).sum()} normal')
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print(f'  Accuracy  : {accuracy_score(gt_labels, preds):.4f}')
print(f'  Precision : {precision_score(gt_labels, preds):.4f}')
print(f'  Recall    : {recall_score(gt_labels, preds):.4f}')
print(f'  F1        : {f1_score(gt_labels, preds):.4f}')
print(f'  ROC-AUC   : {roc_auc_score(gt_labels, gate_scores):.4f}')
print()
print(classification_report(gt_labels, preds, target_names=['Normal', 'Anomalous']))

In [ ]:
# ── Cell 15: Download results to local machine ───────────────────────────────
# Run this after judge step to save all result files locally.
# Files are also always saved to MyDrive/rtfm_pipeline_outputs/ automatically.

import shutil, zipfile
from google.colab import files

# Bundle everything into one zip
zip_path = '/content/rtfm_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for json_file in OUTPUT_DIR.glob('**/*.json'):
        zf.write(json_file, json_file.relative_to(OUTPUT_DIR))

print(f'Zipped {sum(1 for _ in OUTPUT_DIR.glob("**/*.json"))} JSON files')
print('Downloading...')
files.download(zip_path)
print()
print('Unzip locally with:')
print('  unzip rtfm_results.zip -d pipeline/rtfm_colab_outputs/')
